# SLAM 高斯-牛顿定位（notebook 版）

一个通用 Gauss-Newton 引擎，分别完成四道题：

- **Q1 单点定位**：一个机器人、固定位置，随机/给定初值，迭代求出自身坐标 (1,2,1)
- **Q2 轨迹**：一个机器人、4 个时刻，逐时刻定位并连成轨迹，附反验证
- **Q3 群体**：5 个机器人、同一时刻，各自定位后求 5×5 距离矩阵
- **Q4 综合**：5 个机器人 × 200 个时刻，随机 landmark 与轨迹，暖启动求解并做轨迹/相似度可视化

**关于旋转基**：LiDAR 给的是标量距离（向量范数），正交基变换不改变范数 $\|B_L^{-1}v\|=\|v\|$，因此所有题目统一在世界系（$B_w=I$）下表达，不再单独处理 $B_L$。

> 每个需要手动调整的参数，下文都标注了它所在的具体 cell。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False
np.set_printoptions(precision=4, suppress=True)

ROBOT_COLORS = ["#2a9d8f","#e76f51","#e9c46a","#264653","#6a4c93"]
TEAL = LinearSegmentedColormap.from_list("teal", ["#eaf4f2","#0f4c5c"])

## 通用 Gauss-Newton 引擎

残差 $r_i=h_i(X)-d_i$，其中预测距离 $h_i=\|c_i-X\|$；雅可比第 $i$ 行 $=-(c_i-X)/h_i$。
单步返回 R、J、JᵀJ、(JᵀJ)⁻¹、JᵀR、Δ、X_next，SSR 按老师定义 $SSR=\frac12\sum r_i^2$。

In [ ]:
def gn_step(C, X, d):
    U = C - X
    h = np.linalg.norm(U, axis=1)
    R = h - d
    J = -U / h[:, None]
    JtJ = J.T @ J
    JtR = J.T @ R
    delta = -np.linalg.solve(JtJ, JtR)
    return {"R":R, "J":J, "JtJ":JtJ, "JtJinv":np.linalg.inv(JtJ),
            "JtR":JtR, "delta":delta, "Xnext":X+delta, "ssr":0.5*R@R}

def solve_unit(C, d, X0, max_iter=30, tol=1e-8):
    X = np.array(X0, float).copy()
    steps = []
    for _ in range(max_iter):
        st = gn_step(C, X, d)
        steps.append(st)
        X = st["Xnext"]
        if np.linalg.norm(st["delta"]) < tol:
            break
    converged = np.linalg.norm(steps[-1]["delta"]) < tol
    return {"finalX":X, "steps":steps, "converged":converged}

## 绘图工具（3D 轨迹 + 热图）

3D 轨迹中：圆点=起点、方块=终点、黑色点=landmark；本地运行可拖拽旋转。

In [ ]:
def plot_tracks(tracks, landmarks=None, title=""):
    fig = plt.figure(figsize=(9,6))
    ax = fig.add_subplot(111, projection="3d")
    for i, tr in enumerate(tracks):
        tr = np.array(tr); c = ROBOT_COLORS[i % len(ROBOT_COLORS)]
        ax.plot(tr[:,0], tr[:,1], tr[:,2], color=c, lw=2)
        ax.scatter(*tr[0], color=c, marker="o", s=45)
        ax.scatter(*tr[-1], color=c, marker="s", s=45)
    if landmarks is not None:
        L = np.array(landmarks)
        ax.scatter(L[:,0], L[:,1], L[:,2], color="#111", s=35)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(title)
    return fig

def plot_heat(M, labels, title, annotate=True):
    fig, ax = plt.subplots(figsize=(6,5))
    M = np.array(M)
    im = ax.imshow(M, cmap=TEAL, vmin=np.min(M), vmax=np.max(M))
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    if annotate:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, format(M[i,j], ".3g"), ha="center", va="center",
                        color="#fff" if M[i,j] > (M.min()+M.max())/2 else "#16303a", fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    return fig

---
# Q1 单点定位

**改初值、landmark 或测距：只改下面这个 cell**（例如把 X0 改成随机点 `np.random.uniform(-3,3,3)`）。

In [ ]:
C1 = np.array([[4,0,0],
               [0,5,0],
               [0,0,6],
               [4,5,6]], float)
d1 = np.array([np.sqrt(14), np.sqrt(11), np.sqrt(30), np.sqrt(43)])
X0 = np.zeros(3)

q1 = solve_unit(C1, d1, X0)
print("收敛:", q1["converged"])
print("最终位置 X =", q1["finalX"])
for k, st in enumerate(q1["steps"]):
    print(f"第{k+1}轮  X={st['Xnext']}  SSR={round(st['ssr'],6)}")

### Q1 第一步的全套矩阵（X[0] → X[1]）

In [ ]:
s0 = q1["steps"][0]
print("X[0] =", X0)
for key in ["R","J","JtJ","JtJinv","JtR","delta","Xnext"]:
    print(key, "=\n", s0[key], "\n")

### Q1 反验证

把最终位置代回算预测距离，对照实测，残差应全部接近 0。

In [ ]:
pred = np.linalg.norm(C1 - q1["finalX"], axis=1)
print("预测距离:", pred)
print("实测距离:", d1)
print("残差:", pred - d1)

---
# Q2 轨迹（单机器人 × 4 时刻）

**改 4 个时刻的测距：只改下面这个 cell**（每行一个时刻、按 A/B/C/D 顺序）。初值固定为 0，之后沿时间暖启动。

In [ ]:
C2 = C1.copy()
d2 = np.array([
 [3.741657, 3.316625, 5.477226, 6.557439],
 [3.690528, 3.036445, 5.900847, 6.574192],
 [3.648287, 3.051229, 5.874521, 6.205643],
 [13.80507, 15.98687, 19.03103, 21.24100]])

track2, prev = [], np.zeros(3)
for t in range(len(d2)):
    u = solve_unit(C2, d2[t], prev)
    track2.append(u["finalX"]); prev = u["finalX"]
track2 = np.array(track2)
print(track2)

### Q2-2 反验证 + 轨迹图

In [ ]:
for t in range(len(d2)):
    pred = np.linalg.norm(C2 - track2[t], axis=1)
    print(f"t{t} 残差:", np.round(pred - d2[t], 6))
plot_tracks([track2], C2, "Q2 单机器人轨迹（圆=起点 方=终点）")
plt.show()

---
# Q3 群体定位（5 机器人 × 同一时刻）

**改 landmark 或 5 组测距：只改下面这个 cell**。每个机器人独立从 0 开始定位（单时刻无时间延续），再求两两距离矩阵。

In [ ]:
C3 = np.array([[-3,4,-7],
               [-21,14,0],
               [9,7,-18],
               [13,17,3]], float)
d3 = np.array([
 [18.41195, 35.58089, 24.18677, 20.24846],
 [8.944272, 18.574176, 23.874673, 31.064449],
 [7.81025, 24.37212, 13.37909, 16.43168],
 [9.273618, 20.712315, 25.019992, 28.757608],
 [20.34699, 31.92178, 29.76575, 19.20937]])

pos3 = np.array([solve_unit(C3, d3[i], np.zeros(3))["finalX"] for i in range(5)])
D3 = np.zeros((5,5))
for i in range(5):
    for j in range(5):
        D3[i,j] = np.linalg.norm(pos3[i]-pos3[j])
print("各机器人位置:\n", pos3)
print("5×5 距离矩阵:\n", D3)
plot_heat(D3, [f"R{i+1}" for i in range(5)], "Q3 机器人两两距离矩阵")
plt.show()

---
# Q4 综合（5 机器人 × 200 时刻）

**改采样规模、时间步长或随机种子：只改下面这个 cell 开头的 T / dt / seed**。
landmark 沿四面体四个对角方向生成（保证不共面、从四周包围），轨迹由匀速+正弦摆动生成，求解采用暖启动。

In [ ]:
T = 200          # 时刻总数
dt = 0.1         # 时间步长
seed = 42        # 随机种子（改这个可换一批数据）
rng = np.random.default_rng(seed)

tetra = np.array([[1,1,1],[1,-1,-1],[-1,1,-1],[-1,-1,1.]]) / np.sqrt(3)
C4 = np.array([np.round(d*rng.uniform(18,24) + rng.uniform(-1.5,1.5,3)) for d in tetra])

def gen_track(rng):
    p0 = rng.uniform(-9,9,3); v = rng.uniform(-0.7,0.7,3)
    A = rng.uniform(-3,3,3); w = rng.uniform(0.2,0.45); ph = rng.uniform(0,6.28)
    ts = np.arange(T)*dt
    return np.array([p0 + v*t + A*np.sin(w*t+ph) for t in ts])

true4 = [gen_track(rng) for _ in range(5)]
meas4 = [[np.linalg.norm(C4 - true4[i][k], axis=1) for k in range(T)] for i in range(5)]

solved4 = []
for i in range(5):
    tr, prev = [], np.zeros(3)
    for k in range(T):
        u = solve_unit(C4, meas4[i][k], prev)
        tr.append(u["finalX"]); prev = u["finalX"]
    solved4.append(np.array(tr))
print("求解完成，5 机器人 ×", T, "时刻")

### Q4-1 指定时刻的距离矩阵

**想看其他时刻：只改下面这个 cell 里的 pick_t**（取值 0 到 T-1）。每个时刻对应一个 5×5 距离矩阵。

In [ ]:
pick_t = 0
D4 = np.zeros((5,5))
for i in range(5):
    for j in range(5):
        D4[i,j] = np.linalg.norm(solved4[i][pick_t]-solved4[j][pick_t])
print(D4)
plot_heat(D4, [f"R{i+1}" for i in range(5)], f"t={round(pick_t*dt, 6)} 距离矩阵")
plt.show()

### Q4-2 真实轨迹 vs 反解轨迹

实线=真实轨迹，虚线=反解轨迹；黑色点=landmark。本地可拖拽旋转。

In [ ]:
fig = plt.figure(figsize=(10,7)); ax = fig.add_subplot(111, projection="3d")
for i in range(5):
    c = ROBOT_COLORS[i]
    ax.plot(true4[i][:,0], true4[i][:,1], true4[i][:,2], color=c, lw=2)
    ax.plot(solved4[i][:,0], solved4[i][:,1], solved4[i][:,2], color=c, lw=1, ls="--")
ax.scatter(C4[:,0], C4[:,1], C4[:,2], color="#111", s=35)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title("实线=真实 虚线=反解")
plt.show()

### Q4-3 各机器人轨迹（分面）

每个机器人单独一个 3D 视图，展示其完整轨迹（圆=起点、方=终点），便于逐个查看、互不遮挡。

In [ ]:
fig = plt.figure(figsize=(13,8))
for i in range(5):
    ax = fig.add_subplot(2,3,i+1, projection="3d")
    tr = solved4[i]; c = ROBOT_COLORS[i]
    ax.plot(tr[:,0], tr[:,1], tr[:,2], color=c, lw=2)
    ax.scatter(*tr[0], color=c, marker="o", s=35)
    ax.scatter(*tr[-1], color=c, marker="s", s=35)
    ax.scatter(C4[:,0], C4[:,1], C4[:,2], color="#111", s=15)
    ax.set_title(f"R{i+1}"); ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
fig.tight_layout()
plt.show()

### Q4-4 方向相似度 cosθ

对每条轨迹的逐时刻速度方向求平均夹角余弦：越接近 1 越同向，越接近 -1 越反向。
热图用同 teal 色相的深浅（取绝对值）着色，正负请读格子里的真实数字。

In [ ]:
def cosine_matrix(tracks):
    n = len(tracks); vel = [np.diff(tr, axis=0) for tr in tracks]
    M = np.eye(n)
    for i in range(n):
        for j in range(i+1,n):
            vals = []
            for k in range(len(vel[i])):
                a, b = vel[i][k], vel[j][k]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 1e-9 and nb > 1e-9:
                    vals.append(a@b/(na*nb))
            M[i,j] = M[j,i] = np.mean(vals)
    return M

cosM = cosine_matrix(solved4)
labs = [f"R{i+1}" for i in range(5)]
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(np.abs(cosM), cmap=TEAL, vmin=0, vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels(labs)
ax.set_yticks(range(5)); ax.set_yticklabels(labs)
for i in range(5):
    for j in range(5):
        ax.text(j, i, format(cosM[i,j], ".3f"), ha="center", va="center",
                color="#fff" if abs(cosM[i,j])>0.55 else "#16303a", fontsize=9)
ax.set_title("cosθ 方向相似度（同色相深浅）")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()
bi, bj = np.unravel_index(np.argmax(cosM - np.eye(5)), cosM.shape)
print(f"最同向的两条轨迹：R{bi+1} 与 R{bj+1}，cosθ={cosM[bi,bj]:.4f}")

## 小结：参数都在哪个 cell

- 改 Q1 初值/landmark/测距：**Q1 输入 cell**（变量 C1、d1、X0）
- 改 Q2 四时刻测距：**Q2 数据 cell**（变量 d2）
- 改 Q3 landmark/测距：**Q3 数据 cell**（变量 C3、d3）
- 改 Q4 规模/步长/随机种子：**Q4 参数 cell**（变量 T、dt、seed）
- 看 Q4 其他时刻的距离矩阵：**Q4-1 cell**（变量 pick_t）
- 各题统一在世界系（Bw=I）下计算；收敛时 SSR→0、Δ→0。